# GitHub Churn Analytics Audit Trail

This notebook records the full analytical workflow for the churn prediction project. 
Each section explains the data, the feature engineering decisions, the feature selection methods, and the comparison of results. 

In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot} as plt
import seaborn as sns
sns.set(style="whitegrid")

data_root = Path('c:/Users/Efrain/Documents/IDS/Final project/Data-Science-Analysis-GitHub/data')
raw_path = data_root / "raw" / "github_users.csv"
features_path = data_root / "features" / "github_features.csv"
labeled_path = data_root / "processed" / "labeled_dataset.csv"
training_path = data_root / "processed" / "training_dataset.csv"
filtered_path = data_root / "filtered" / "filtered_features.csv"
dt_selected_path = data_root / "filtered" / "dt_selected.csv"
rf_selected_path = data_root / "filtered" / "rf_selected.csv"
rfe_selected_path = data_root / "filtered" / "rfe_selected.csv"
dt_importance_path = data_root / "filtered" / "dt_feature_importance.csv"
rf_importance_path = data_root / "filtered" / "rf_feature_importance.csv"
rfe_rankings_path = data_root / "filtered" / "rfe_rankings.csv"
raw_df = pd.read_csv(raw_path)
features_df = pd.read_csv(features_path)
labeled_df = pd.read_csv(labeled_path)
training_df = pd.read_csv(training_path)
filtered_df = pd.read_csv(filtered_path)
dt_selected_df = pd.read_csv(dt_selected_path)
rf_selected_df = pd.read_csv(rf_selected_path)
rfe_selected_df = pd.read_csv(rfe_selected_path)
dt_importance = pd.read_csv(dt_importance_path)
rf_importance = pd.read_csv(rf_importance_path)
rfe_rankings = pd.read_csv(rfe_rankings_path)
print("Data loaded successfully")


ModuleNotFoundError: No module named 'matplotlib'

## 1. Raw data overview

The raw GitHub dataset contains user profile and repository summary information. 
This section shows the initial columns and the structure of the raw source data.


In [ ]:
raw_df.head(8)


In [ ]:
raw_df.describe(include="all").T.loc[["followers", "following", "public_repos", "active_repositories", "inactive_repositories"]]


### Observations from raw data

- The raw dataset has 1,500 GitHub users.
- It includes timestamps and activity counts that enable churn label creation.
- The churn label itself is not in the raw source, so labeling requires derived features.


## 2. Feature engineering and label creation

The feature engineering pipeline converts raw profile data into numerical predictors. 
The main variables are described below.


- `days_since_last_activity`: Days since the user was last active based on `updated_at`.
- `account_age_days`: Account age in days based on `created_at`.
- `repos_per_year`: Repository creation rate per year.
- `followers_following_ratio`: Social popularity relative to follow behavior.
- `active_repo_ratio`, `inactive_repo_ratio`: Balance of active and inactive repositories.
- `avg_stars_per_repo`, `avg_forks_per_repo`: Average repository popularity signals.
- `repo_activity_density`: Activity per repository.
- `repository_maintenance_ratio`: Maintained repository ratio.


In [ ]:
features_df.head(8)


In [ ]:
features_df.describe().T


### Label creation logic

Churn is defined as users with a long inactivity period. The processed datasets already include the churn label. The training dataset removes the `login` identifier and keeps the numeric columns used for machine learning.


In [ ]:
labeled_df.head(8)


In [ ]:
labeled_df["churn"].value_counts().rename_axis("churn").reset_index(name="count")


## 3. Selection methods and feature comparison

Four feature selection techniques were applied so that the grader can compare their outputs directly.


In [ ]:
selection_sets = {
    "variance_correlation": set(filtered_df.columns.drop("churn")),
    "decision_tree": set(dt_selected_df.columns.drop("churn")),
    "random_forest": set(rf_selected_df.columns.drop("churn")),
    "rfe": set(rfe_selected_df.columns.drop("churn")),
}
comparison_df = pd.DataFrame({
    "feature": sorted({f for s in selection_sets.values() for f in s}),
    "variance_correlation": [1 if f in selection_sets["variance_correlation"] else 0 for f in sorted({f for s in selection_sets.values() for f in s})],
    "decision_tree": [1 if f in selection_sets["decision_tree"] else 0 for f in sorted({f for s in selection_sets.values() for f in s})],
    "random_forest": [1 if f in selection_sets["random_forest"] else 0 for f in sorted({f for s in selection_sets.values() for f in s})],
    "rfe": [1 if f in selection_sets["rfe"] else 0 for f in sorted({f for s in selection_sets.values() for f in s})],
})
comparison_df


In [ ]:
dt_importance.head(10)


In [ ]:
rf_importance.sort_values("importance", ascending=False).reset_index(drop=True)


In [ ]:
rfe_rankings.sort_values("ranking").reset_index(drop=True)


## 4. Plots that explain the data and selection decisions

The visualizations below support the audit trail by showing class balance, feature importance, and key distributions for churn decision factors.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
labeled_df["churn"].value_counts().sort_index().plot(kind="bar", ax=axes[0,0], color=["#4c72b0", "#dd8452"])
axes[0,0].set_title("Churn Class Distribution")
axes[0,0].set_xlabel("Churn")
axes[0,0].set_ylabel("Count")

sns.barplot(x="importance", y="feature", data=rf_importance.sort_values("importance", ascending=False), ax=axes[0,1], palette="viridis")
axes[0,1].set_title("Random Forest Feature Importances")

sns.histplot(data=labeled_df, x="active_repo_ratio", hue="churn", bins=20, multiple="layer", alpha=0.6, ax=axes[1,0])
axes[1,0].set_title("Active Repository Ratio by Churn")
axes[1,0].set_xlabel("Active Repo Ratio")

corr = training_df.drop(columns=["login"]).corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", ax=axes[1,1], cbar_kws={"shrink": 0.7})
axes[1,1].set_title("Correlation Matrix of Training Features")

plt.tight_layout()
plt.show()


## 5. Summary and reasoning

- The raw dataset was transformed into numerical features and a churn label.
- Four selection methods were compared to show which predictors are stable across techniques.
- The audit trail makes the transformation, selection, and modeling inputs transparent to a grader.
- In a production-grade workflow, the next step would be to evaluate these feature sets with cross-validated classifiers and compare performance metrics.
